In [1]:
library(dplyr)
library(Seurat)
library(future)

plan("multicore", workers = 24)
options(future.globals.maxSize = 100000 * 1024^3)

set.seed(1234)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t




In [2]:
primary <- readRDS('/projects/0/einf2548/cruiz/dmg/notebooks/scATAC/data/ascites_case/seurat-mnn_integration_sct-norm_primary-tumor_alternative.rds')
primary

An object of class Seurat 
48434 features across 5422 samples within 2 assays 
Active assay: SCT (24217 features, 3000 variable features)
 3 layers present: counts, data, scale.data
 1 other assay present: RNA
 2 dimensional reductions calculated: mnn, umap

In [3]:
ascites <- readRDS('/projects/0/einf2548/cruiz/dmg/notebooks/scATAC/data/ascites_case/seurat-mnn_integration_sct-norm_all-prettx_no-low-complex-cells.rds')
ascites

An object of class Seurat 
44017 features across 21704 samples within 2 assays 
Active assay: SCT (21525 features, 3000 variable features)
 3 layers present: counts, data, scale.data
 1 other assay present: RNA
 2 dimensional reductions calculated: mnn, umap

In [6]:
process_seurat_object <- function(seurat_obj, orig_ident_names) {
  # Subset Seurat object by orig.ident (with multiple values)
  subset_obj <- subset(seurat_obj, subset = orig.ident %in% orig_ident_names)
  
  # Extract barcodes and clean up
  barcodes <- colnames(subset_obj)
  cleaned_barcodes <- gsub("^.*_", "", barcodes)
  
  # Save barcodes to TSV
  for (ident in orig_ident_names) {
    ident_barcodes <- cleaned_barcodes[grepl(paste0("^", ident), barcodes)]
    output_filename <- paste0(ident, "_barcodes.tsv")
    write.table(ident_barcodes, output_filename, sep = "\t", row.names = FALSE, col.names = FALSE, quote = FALSE)
  }
}


orig_ident_primary <- c('core', 'edge')
orig_ident_ascites <- c('first','second', 'third')

# Process the primary object
process_seurat_object(primary, orig_ident_primary)

# Process the ascites object
process_seurat_object(ascites, orig_ident_ascites)